# Mock 5A · Solar Filament Segmentation · Phân đoạn sợi plasma (filament) trên ảnh Mặt Trời (Computer Vision · Thị giác máy tính)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | Each image is a full-disk photo of the Sun in the H-alpha line from the GONG telescope network. **Filaments** are clouds of cooler plasma held above the surface by magnetic fields; they look like thin dark threads on the disc. Find **every filament as a separate object** (instance segmentation): one mask per filament. | Mỗi ảnh là ảnh toàn đĩa Mặt Trời ở vạch H-alpha do mạng kính thiên văn GONG chụp. **Filament** là những đám plasma nguội hơn được từ trường giữ lơ lửng; trên đĩa chúng trông như những sợi tối, mảnh. Hãy tìm **từng filament như một đối tượng riêng** (instance segmentation): mỗi filament một mặt nạ. |
| **Data** | The original data of the Kaggle *Solar Filament Segmentation Challenge 2026*. `train/` 467 images (observed 2011-2015) + `annotations.json` (COCO format, exactly as on Kaggle: one polygon per filament) · `public_test/` (121) and `private_test/` (119) images, observed **2016-2022**: later years, a quieter phase of the solar cycle. Images are **2048×2048** greyscale. Some images were annotated by 2-3 people independently: each person is one **view** (`v1-<id>`, `v2-<id>`, ...). | Dữ liệu gốc của cuộc thi Kaggle *Solar Filament Segmentation Challenge 2026*. `train/` 467 ảnh (chụp 2011-2015) + `annotations.json` (định dạng COCO, giống hệt Kaggle: mỗi filament một đa giác) · `public_test/` (121) và `private_test/` (119) ảnh, chụp **2016-2022**: những năm sau, pha Mặt Trời yên tĩnh hơn. Ảnh **2048×2048**, ảnh xám. Một số ảnh được 2-3 người gán nhãn độc lập: mỗi người là một **view** (`v1-<id>`, `v2-<id>`, ...). |
| **Metric** | **Panoptic Quality** (the Kaggle metric). In every view a predicted filament matches a true one when **IoU > 0.5**; TP = matches, FP = predicted filaments that match nothing, FN = true filaments that nothing matches; counted over all views and images: PQ = ΣIoU(TP) / (TP + ½FP + ½FN). **Higher is better.** Ranking uses the **private** score. | **Panoptic Quality** (độ đo của Kaggle). Trong mỗi view, một filament dự đoán khớp với filament thật khi **IoU > 0.5**; TP = cặp khớp, FP = filament dự đoán không khớp gì, FN = filament thật không được khớp; cộng dồn trên mọi view và ảnh: PQ = ΣIoU(TP) / (TP + ½FP + ½FN). **Càng cao càng tốt.** Xếp hạng theo điểm **private**. |
| **Submission** | Kaggle format: `public_submission.csv`, `private_submission.csv` with columns `filament_id,segmentation_rle`, **one row per predicted filament**: `filament_id` = `<image id>_<k>`, `segmentation_rle` = **compressed COCO RLE** of that filament's mask at **2048×2048** (`mask_to_rle` below = `pycocotools.mask.encode(...)["counts"]`). An image with no row = no filament predicted. | Định dạng Kaggle: `public_submission.csv`, `private_submission.csv` gồm cột `filament_id,segmentation_rle`, **mỗi filament dự đoán một dòng**: `filament_id` = `<id ảnh>_<k>`, `segmentation_rle` = **COCO RLE nén** của mặt nạ filament đó ở **2048×2048** (`mask_to_rle` bên dưới = `pycocotools.mask.encode(...)["counts"]`). Ảnh không có dòng nào = không dự đoán filament. |
| **Rules** | PyTorch. ImageNet-pretrained backbones from `torchvision` / `timm` are allowed. **No external solar data**: in particular **do not use the Kaggle competition files or the MAGFiLO dataset** (the test images are labelled there: honour code). No hand-labelling of test images. | PyTorch. Được dùng backbone huấn luyện sẵn trên ImageNet từ `torchvision` / `timm`. **Không dùng dữ liệu Mặt Trời bên ngoài**, đặc biệt **không dùng file của cuộc thi Kaggle hay bộ MAGFiLO** (ảnh test có nhãn ở đó: hãy giữ danh dự). Không tự gán nhãn ảnh test. |

$$\text{PQ}=\frac{\sum_{(p,g)\in TP}\text{IoU}(p,g)}{|TP|+\tfrac12|FP|+\tfrac12|FN|},\qquad (p,g)\in TP \iff \text{IoU}(p,g)>0.5$$

**Files · Các file:**
- `train/images/<id>.jpeg` (2048×2048), `train/annotations.json` — COCO: `images` (one entry per view · mỗi view một mục, `id` = `v<k>-<image id>`), `annotations` (`image_id` = view, `segmentation` = list of polygons [x0, y0, x1, y1, ...] in 2048-pixel coordinates · danh sách đa giác, toạ độ theo ảnh 2048; also `bbox`, `spine`, `category_id` = chirality · chiều xoắn, not needed for the metric · không dùng trong độ đo)
- `train/train.csv` — `id, site, year, n_annotators, n_filaments`
- `public_test/images/`, `public_test/public_test.csv`, `private_test/images/`, `private_test/private_test.csv` — `id, site`

🇬🇧 The metric counts **objects**: a filament broken into two pieces, or two filaments merged into one blob, usually scores 0 for those filaments, and every stray dark speck you submit is a false positive. Sunspots and dark patches near active regions are not filaments. The annotators themselves disagree on faint filaments and on where one filament ends.
🇻🇳 Độ đo tính theo **đối tượng**: một filament bị vỡ làm hai mảnh, hoặc hai filament bị dính thành một khối, thường được 0 điểm cho các filament đó, và mỗi đốm tối lẻ bạn nộp là một false positive. Vết đen Mặt Trời và các mảng tối quanh vùng hoạt động không phải filament. Chính người gán nhãn cũng không thống nhất với những sợi mờ và chỗ một filament kết thúc.

Data · Dữ liệu: [Kaggle Solar Filament Segmentation Challenge 2026](https://www.kaggle.com/competitions/filament-segmentation-2026) training set: images GONG H-alpha, NSO/AURA/NSF (CC BY 4.0); annotations [MAGFiLO v1.0](https://doi.org/10.7910/DVN/J6JNVK), Ahmadzadeh et al. 2024 (CC BY-NC 4.0). See `dataset/README.md`.
Inspired by · Lấy cảm hứng từ: [IEEE BigData 2026 Solar Filament Segmentation (Kaggle)](https://www.kaggle.com/competitions/filament-segmentation-2026).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import json
import time

import cv2
import matplotlib.pyplot as plt

try:
    from pycocotools import mask as mask_utils  # preinstalled on Colab; the official Kaggle evaluation uses it
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)
    from pycocotools import mask as mask_utils
import torch.nn.functional as F
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "5A"
FOLDER  = "5A_solar_filament"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/mock-5"

# Already have the folder? Point this at the parent of 5A_solar_filament/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 5A_solar_filament/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/5A_solar_filament/dataset"
TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
TRAIN_JSON = f"{DATA_DIR}/train/annotations.json"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
SIZE = 2048  # images and submitted masks / ảnh và mặt nạ nộp bài
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
IMG_SIZE = 512    # training/prediction resolution (images are 2048) / độ phân giải khi huấn luyện, dự đoán
BASE_CH = 16      # channels of the first U-Net level / số kênh tầng đầu của U-Net
BATCH_SIZE = 8
EPOCHS = 12
LR = 1e-3
THRESHOLD = 0.7   # probability -> mask (POS_WEIGHT makes probabilities run high) / ngưỡng xác suất -> mặt nạ
POS_WEIGHT = 10.0  # filaments are < 1% of the pixels: weight them up in the loss / filament < 1% điểm ảnh nên tăng trọng số
MIN_AREA = 300    # drop predicted objects smaller than this (pixels at 2048) / bỏ đối tượng nhỏ hơn (điểm ảnh ở 2048)
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Visualisation · Đọc và hiển thị dữ liệu

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
public_df = pd.read_csv(PUBLIC_CSV)
private_df = pd.read_csv(PRIVATE_CSV)
coco = json.load(open(TRAIN_JSON))
print(f"Train: {len(train_df)} images, {len(coco['images'])} views, {len(coco['annotations'])} filaments "
      f"| Public test: {len(public_df)} | Private test: {len(private_df)}")

# EN: views[image id][view id] = list of filaments, each a list of polygons [x0, y0, x1, y1, ...] (2048-px coordinates)
# VI: views[id ảnh][id view] = danh sách filament, mỗi filament là danh sách đa giác [x0, y0, x1, y1, ...]
views = {}
for im in coco["images"]:
    views.setdefault(im["file_name"].rsplit(".", 1)[0], {})[im["id"]] = []
for a in coco["annotations"]:
    image_id = a["image_id"].split("-", 1)[1]
    views[image_id][a["image_id"]].append(a["segmentation"])


def filament_mask(polygons, size=SIZE):
    """EN: one filament (list of polygons in 2048-px coordinates) -> boolean mask size x size, drawn with pycocotools
    exactly like the hidden test masks. VI: một filament (danh sách đa giác, toạ độ 2048) -> mặt nạ size x size,
    vẽ bằng pycocotools giống hệt mặt nạ test ẩn."""
    polys = [list(np.asarray(p, np.float64) * size / SIZE) for p in polygons if len(p) >= 6]
    return mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(polys, size, size))).astype(bool)


def target_mask(image_id, size=IMG_SIZE):
    """EN: training target: a pixel is filament if at least half of the views drew it.
    VI: nhãn huấn luyện: điểm ảnh là filament nếu ít nhất một nửa số view vẽ nó."""
    per_view = [np.any([filament_mask(f, size) for f in fils], axis=0) if fils else np.zeros((size, size), bool)
                for fils in views[image_id].values()]
    return (np.sum(per_view, axis=0) >= (len(per_view) + 1) // 2).astype(np.float32)


# ---- COCO RLE (same strings as pycocotools.mask.encode(np.asfortranarray(mask))["counts"].decode()) ----
def _counts_to_string(counts):
    out = []
    for i, c in enumerate(counts):
        x = int(c) - (int(counts[i - 2]) if i > 2 else 0)
        more = True
        while more:
            ch = x & 0x1F
            x >>= 5
            more = (x != -1) if (ch & 0x10) else (x != 0)
            out.append(chr((ch | 0x20 if more else ch) + 48))
    return "".join(out)


def mask_to_rle(pixels):
    """EN: ONE filament -> compressed COCO RLE string at 2048x2048. `pixels` = boolean mask (2048, 2048) or the sorted
    column-major pixel indices (x * 2048 + y) of the filament.
    VI: MỘT filament -> chuỗi COCO RLE nén ở 2048x2048. `pixels` = mặt nạ (2048, 2048) hoặc chỉ số điểm ảnh theo cột."""
    idx = np.flatnonzero(np.asarray(pixels).ravel(order="F")) if np.ndim(pixels) == 2 else np.asarray(pixels)
    breaks = np.flatnonzero(np.diff(idx) > 1)
    starts = np.concatenate([[idx[0]], idx[breaks + 1]])
    ends = np.concatenate([idx[breaks], [idx[-1]]]) + 1
    counts, prev = [], 0
    for a, b in zip(starts, ends):  # alternating runs of background / filament
        counts += [int(a - prev), int(b - a)]
        prev = b
    if prev < SIZE * SIZE:
        counts.append(int(SIZE * SIZE - prev))
    return _counts_to_string(counts)


def load_images(split, ids, size=IMG_SIZE):
    """EN: read the 2048x2048 greyscale images and shrink them to size x size -> uint8 (N, size, size).
    VI: đọc ảnh xám 2048x2048 và thu nhỏ về size x size -> uint8 (N, size, size)."""
    return np.stack([cv2.resize(np.asarray(Image.open(f"{DATA_DIR}/{split}/images/{i}.jpeg").convert("L")),
                                (size, size), interpolation=cv2.INTER_AREA) for i in ids])


t0 = time.time()
X_all = load_images("train", train_df["id"])
Y_all = np.stack([target_mask(i) for i in train_df["id"]])
X_public = load_images("public_test", public_df["id"])
X_private = load_images("private_test", private_df["id"])
print(f"Images in memory / Ảnh trong bộ nhớ: {X_all.shape} ({time.time() - t0:.0f}s), "
      f"filament pixels / điểm ảnh filament: {Y_all.mean():.4%}")

# EN: random 10% validation split / VI: tách ngẫu nhiên 10% làm tập validation
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X_all))
n_val = len(X_all) // 10
val_idx, tr_idx = perm[:n_val], perm[n_val:]


def to_tensor(x):
    return torch.from_numpy(x).float().div(255.0).unsqueeze(1)  # (N, 1, H, W) in [0, 1]


train_loader = DataLoader(TensorDataset(to_tensor(X_all[tr_idx]), torch.from_numpy(Y_all[tr_idx]).unsqueeze(1)),
                          batch_size=BATCH_SIZE, shuffle=True)
print(f"Train: {len(tr_idx)} | Validation: {len(val_idx)}")

# EN: a few training images, each filament of the first view in its own colour
# VI: vài ảnh train, mỗi filament (của view đầu tiên) một màu
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, i in zip(axes, rng.choice(len(X_all), 3, replace=False)):
    image_id = train_df["id"][i]
    rgb = np.repeat(X_all[i][..., None], 3, axis=2).copy()
    first_view = next(iter(views[image_id].values()))
    for f in first_view:
        rgb[filament_mask(f, IMG_SIZE)] = rng.integers(80, 256, 3)
    ax.imshow(rgb)
    ax.set_title(f"{image_id} · {train_df['year'][i]} · {len(first_view)} filaments · {len(views[image_id])} view(s)")
    ax.axis("off")
plt.show()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
def block(c_in, c_out):
    """EN: two 3x3 convolutions with batch norm. / VI: hai lớp tích chập 3x3 kèm batch norm."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """EN: small U-Net trained from scratch: 3 down-sampling steps, skip connections, 1 output logit per pixel.
    It only says "filament or not" for each pixel; separate filaments are found afterwards (connected components).
    VI: U-Net nhỏ huấn luyện từ đầu: 3 lần giảm kích thước, kết nối tắt, 1 logit cho mỗi điểm ảnh.
    Mô hình chỉ nói "filament hay không" cho từng điểm ảnh; các filament riêng được tách sau (thành phần liên thông)."""

    def __init__(self, c=BASE_CH):
        super().__init__()
        self.enc1, self.enc2, self.enc3 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c)
        self.mid = block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), block(2 * c, c)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        m = self.mid(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([self.up3(m), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)


model = UNet().to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict_instances(model, x_uint8, threshold=THRESHOLD, min_area=MIN_AREA):
    """EN: images (N, IMG_SIZE, IMG_SIZE) -> for each image a list of filaments, each given by its pixel indices at
    2048x2048 (column-major: x * 2048 + y, like COCO RLE): probabilities resized to 2048, thresholded, then every
    8-connected blob of >= min_area pixels = one filament.
    VI: ảnh (N, IMG_SIZE, IMG_SIZE) -> mỗi ảnh một danh sách filament, mỗi filament là chỉ số điểm ảnh ở 2048x2048
    (theo cột: x * 2048 + y): phóng xác suất về 2048, cắt ngưỡng, mỗi vùng liên thông >= min_area điểm ảnh = một filament."""
    model.eval()
    result = []
    for i in range(len(x_uint8)):
        prob = torch.sigmoid(model(to_tensor(x_uint8[i:i + 1]).to(DEVICE)))
        prob = F.interpolate(prob, size=(SIZE, SIZE), mode="bilinear", align_corners=False)[0, 0]
        binary = (prob > threshold).cpu().numpy().astype(np.uint8)
        n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
        objects = []
        for k in range(1, n):
            if stats[k, cv2.CC_STAT_AREA] < min_area:
                continue
            x0, y0, w, h = stats[k, :4]
            yy, xx = np.nonzero(labels[y0:y0 + h, x0:x0 + w] == k)
            objects.append(np.sort((xx + x0) * SIZE + (yy + y0)))
        result.append(objects)
    return result


def panoptic_quality(pred_per_image, gt_per_image):
    """EN: the competition metric. pred_per_image[i] = predicted filaments (pixel indices) of image i;
    gt_per_image[i] = {view id: true filaments (pixel indices)} of image i.
    VI: độ đo của cuộc thi, giống hệt bảng xếp hạng."""
    iou_sum, tp, fp, fn = 0.0, 0, 0, 0
    for preds, gt_views in zip(pred_per_image, gt_per_image):
        for gts in gt_views.values():
            iou = np.zeros((len(gts), len(preds)))
            for a, g in enumerate(gts):
                for b, p in enumerate(preds):
                    inter = len(np.intersect1d(g, p, assume_unique=True))
                    if inter:
                        iou[a, b] = inter / (len(g) + len(p) - inter)
            hit = iou > 0.5
            iou_sum += iou[hit].sum()
            tp += int(hit.sum())
            fp += int((hit.sum(axis=0) == 0).sum())
            fn += int((hit.sum(axis=1) == 0).sum())
    den = tp + 0.5 * fp + 0.5 * fn
    return iou_sum / den if den else 0.0


# EN: true filaments of the validation images at 2048 (pixel indices, column-major), computed once
# VI: filament thật của ảnh validation ở 2048 (chỉ số điểm ảnh theo cột), tính một lần
val_gt = [{v: [np.flatnonzero(filament_mask(f).ravel(order="F")) for f in fils] for v, fils in views[i].items()}
          for i in train_df["id"].iloc[val_idx]]
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))  # logits / mô hình trả về logit
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_pq = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, t0 = 0.0, time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    val_pq = panoptic_quality(predict_instances(model, X_all[val_idx]), val_gt)
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(tr_idx):.4f} | Val PQ={val_pq:.4f} | {time.time() - t0:.0f}s"
    if val_pq > best_pq:
        best_pq = val_pq
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation PQ: {best_pq:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(test_df, split, csv_path):
    """Predict, save <filament_id, segmentation_rle> (one row per filament, 2048x2048 COCO RLE) and zip it.
    Dự đoán, lưu CSV (mỗi filament một dòng, COCO RLE ở 2048x2048) và nén zip."""
    X = X_public if split == "public" else X_private
    rows = []
    for image_id, masks in zip(test_df["id"], predict_instances(model, X)):
        rows += [(f"{image_id}_{k}", mask_to_rle(m)) for k, m in enumerate(masks)]  # m = pixel indices
    sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    sub.to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} ({len(sub)} filaments in {len(test_df)} images) and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df, "public", "public_submission.csv")
export(private_df, "private", "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def rle_area(s):
    """Decode a compressed COCO RLE string -> (total pixels covered, filament pixels). / Giải mã để kiểm tra."""
    counts, p = [], 0
    while p < len(s):
        x, k, more = 0, 0, True
        while more:
            c = ord(s[p]) - 48
            assert 0 <= c <= 63, f"invalid character {s[p]!r}: not a COCO RLE string / không phải chuỗi COCO RLE"
            x |= (c & 0x1F) << (5 * k)
            more = bool(c & 0x20)
            p += 1
            k += 1
            if not more and (c & 0x10):
                x |= -1 << (5 * k)
        counts.append(x + (counts[-2] if len(counts) > 2 else 0))
    return sum(counts), sum(counts[1::2])


def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path, dtype=str)
    assert list(sub.columns) == ["filament_id", "segmentation_rle"], f"{csv_path}: columns must be filament_id,segmentation_rle"
    assert not sub["filament_id"].duplicated().any(), f"{csv_path}: duplicate filament_id"
    image_ids = sub["filament_id"].str.rsplit("_", n=1).str[0]
    assert image_ids.isin(set(test_df["id"])).all(), f"{csv_path}: unknown image id in filament_id (<image id>_<k>)"
    for fid, rle in zip(sub["filament_id"], sub["segmentation_rle"]):
        assert isinstance(rle, str) and rle, f"{csv_path}: {fid}: empty mask, drop the row / mặt nạ rỗng, hãy bỏ dòng"
        total, fg = rle_area(rle)
        assert total == SIZE * SIZE, f"{csv_path}: {fid}: mask must be {SIZE}x{SIZE} / mặt nạ phải {SIZE}x{SIZE}"
        assert fg > 0, f"{csv_path}: {fid}: empty mask, drop the row / mặt nạ rỗng, hãy bỏ dòng"
    print(f"{csv_path}: OK, {len(sub)} filaments, {image_ids.nunique()} of {len(test_df)} images have at least one")


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "5A"          # which problem this notebook is for
CONTEST = "mock-5"    # this notebook is for Mock Contest 5
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")